# Statistical Learning - Homework 4

### NAME:  RAJ SHAH

### NET ID:  ras637

In this homework, we predict whether a recipe is vegetarian or not vegetarian, given the description and ingredients. The data is from https://www.kaggle.com/datasets/shuyangli94/foodcom-recipes-with-search-terms-and-tags. The data has been processed to retain mostly dinner recipes, and sampled to ensure 50/50 vegetarian/non-vegetarian recipes. 

In particular, we compare:
1.  a bag-of-words version, where the input data is a matrix of `recipes x n_ingredients`. Row `i` has a `1` in column `j` if recipe `i` uses ingredient `j`.
2. a DistilBERT version, where the input data is a matrix of `recipes x bert_input`, where `bert_input` is a string containing the recipe title, description and ingredient list.

In [1]:
from transformers import AutoTokenizer, AutoModel
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import TensorDataset, DataLoader
import warnings
import itertools

warnings.filterwarnings('ignore')

### BERT data

- `recipes_processed.csv` is a matrix where each row is a recipe. The column `bert_input` are going to be the input to the DistilBert model

In [2]:
bert_df = pd.read_csv('recipes_processed.csv')

bert_df = bert_df[["bert_input", "vegetarian"]]

print(bert_df.head())
print(bert_df.shape)
print("BERT data shape:", bert_df.shape)

                                          bert_input  vegetarian
0  Soda Bread. Most soda bread recipes I've seen ...        True
1  Romaine &amp; Strawberry Salad. I love this si...        True
2  Lemon Risotto With Garlic Crab. I like differe...       False
3  Black Eyed Pea Casserole. I do not know where ...        True
4  Crawfish Fettuccini. A friend from Thibodaux, ...       False
(14000, 2)
BERT data shape: (14000, 2)


We import the DistilBERT tokenizer and model.

In [3]:
from transformers import logging
logging.set_verbosity_error()

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
mybert = AutoModel.from_pretrained("distilbert-base-uncased")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

### Bag-of-words data

- 'recipes_bow.csv' is the `recipes x vocab_size` matrix, where `vocab_size` is the total number of ingredients
- 'ingredient_dict.csv' is the lookup table for the vocabulary of ingredients

In [4]:
bow_df = pd.read_pickle('recipes_bow.pkl')
ingredient_dict = pd.read_csv('ingredient_dict.csv')

In [5]:
print(bow_df.iloc[:, range(4)].head())
print(ingredient_dict.head())
print(bow_df.shape)
print("BOW data shape:", bow_df.shape)
print("Number of ingredients:", ingredient_dict.shape[0])

   ingred_0  ingred_1  ingred_2  ingred_3
0         1         0         0         0
1         0         0         0         0
2         0         0         0         1
3         1         0         1         0
4         0         0         0         0
     ingred          vocab
0  ingred_0           salt
1  ingred_1      olive oil
2  ingred_2          onion
3  ingred_3  garlic cloves
4  ingred_4          water
(14000, 6055)
BOW data shape: (14000, 6055)
Number of ingredients: 6054


In [6]:
n_vocab = ingredient_dict.shape[0]
bow_feats = ['ingred_'+str(j) for j in range(n_vocab)]

In [7]:
xbow = np.array(bow_df[bow_feats])
ybow = np.array(bow_df['vegetarian'])

### Question 1:

(a) Turn the bag-of-words data into torch tensors.

(b) Turn the strings `bert_input` into tokens for BERT.

In [8]:
# turn bow data into torch tensors (hint: make sure they are of torch.float type)

xbow = torch.tensor(xbow, dtype=torch.float)
ybow = torch.tensor(ybow, dtype=torch.float)

In [9]:
# Turn bert_df['bert_input'] into a list (hint: pandas has an easy function to turn a column into a list)

bert_list = bert_df['bert_input'].tolist()

In [10]:
# turn bert_list into tokens using tokenizer with max_length=180, padding='max_length', truncation=True
bert_tokens = tokenizer(bert_list, max_length=180, padding='max_length', truncation=True, return_tensors='pt')

In [11]:
# get both the input_ids and the attention mask from bert_tokens
bert_ids = bert_tokens['input_ids']
bert_mask = bert_tokens['attention_mask']

In [12]:
# turn bert_df['vegetarian'] into a torch tensor.
ybert = torch.tensor(bert_df['vegetarian'].values, dtype=torch.float)
print("BERT tensor shape:", bert_ids.shape)
print("BOW tensor shape:", xbow.shape)

BERT tensor shape: torch.Size([14000, 180])
BOW tensor shape: torch.Size([14000, 6054])


### Question 2: 

Set up a torch dataset called BERT Dataset. The purpose of this dataset is to have an easy way to access:
- input_ids
- attention_mask
- ybert

In [13]:
class BERTDataset(torch.utils.data.Dataset):
    def __init__(self, input_ids, attention_mask, label):

        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.label = label

    def __len__(self):
        return len(self.label)
    
    def __getitem__(self, idx):
        
        input_ids = self.input_ids[idx]
        attention_mask = self.attention_mask[idx]
        label = self.label[idx]

        return input_ids, attention_mask, label

### Question 3: 

Split both BOW data and BERT data into training and test datasets. You should use the same indices for both datasets (i.e. training examples in BOW are the same training examples in BERT).

In [14]:
n = xbow.shape[0]
split = np.random.binomial(1, p=0.9, size=n) # 1 if training, 0 if test
train_ind = np.where(split == 1)[0]
test_ind  = np.where(split == 0)[0]
print("Train size:", len(train_ind))
print("Test size:", len(test_ind))

Train size: 12583
Test size: 1417


In [15]:
xbow_train = xbow[train_ind]
xbow_test  = xbow[test_ind]

ybow_train = ybow[train_ind]
ybow_test  = ybow[test_ind]

In [16]:
# bert input ids
bert_ids_train = bert_ids[train_ind]
bert_ids_test  = bert_ids[test_ind]

# bert attention mask
bert_mask_train = bert_mask[train_ind]
bert_mask_test  = bert_mask[test_ind]

# bert label
ybert_train = ybert[train_ind]
ybert_test  = ybert[test_ind]

### Question 4: 

(a) Create a neural network with 2 hidden layers for the BOW data to predict vegetarian/non-vegetarian.

(b) Create a neural network which uses a single layer on top of the distilBERT model to predict vegetarian/non-vegetarian.

In [17]:
## neural network model
class BOWNet(nn.Module):
    def __init__(self, input_dim, hidden_dims):
        super(BOWNet, self).__init__()

        # hidden_dims is a 2-dim list: hidden_dims[0], hidden_dims[1]
        # initialize a neural network with linear, ReLU, linear, dropout(p=0.5), ReLU, linear
        # what should the final function be after the last linear layer? (hint: y is {0, 1})

        self.layers = nn.Sequential(
            nn.Linear(input_dim,      hidden_dims[0]),
            nn.ReLU(),
            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.Dropout(p=0.5),
            nn.ReLU(),
            nn.Linear(hidden_dims[1], 1),
            nn.Sigmoid()          # binary output ∈ (0,1)
        )

    def forward(self, x):

        y_pred = self.layers(x).flatten()
        return y_pred
    
    def loss(self, y_pred, y):

        # y is binary {0, 1}. what should the loss function be?
        loss = F.binary_cross_entropy(y_pred, y)

        return loss


In [18]:
class BERTNet(nn.Module):
    def __init__(self):
        super(BERTNet, self).__init__()

        self.bert = mybert # this is the distilbert model from the beginning of the notebook
        self.layer1 = nn.Sequential(nn.Linear(768, 1), 
                                    nn.Sigmoid())

    def forward(self, input_ids, attention_mask):

        bert_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = bert_outputs.last_hidden_state[:, 0, :] # this should get the <CLS> token from distilBERT

        x = self.layer1(pooled_output)

        x = x.squeeze(1)

        return x
    
    def loss(self, y_pred, y):

        loss = F.binary_cross_entropy(y_pred, y)

        return loss

### Question 5:

(a) Create torch datasets and data loaders for the BOW and BERT data.

(b) Instantiate a BERT model and a BOW model.

In [19]:
bert_train  = BERTDataset(bert_ids_train, bert_mask_train, ybert_train)
bert_loader = DataLoader(bert_train, batch_size=50, shuffle=True)

bow_train  = TensorDataset(xbow_train, ybow_train)
bow_loader = DataLoader(bow_train, batch_size=100, shuffle=True)

In [20]:
bert_model = BERTNet()
bow_model  = BOWNet(n_vocab, hidden_dims=[500, 100])

### Question 6:

(a) Train the BERT neural network to predict vegetarian/not vegetarian. 

Note: each epoch may take a while (on Prof Moran's laptop, each epoch took around 20 mins). You can obtain a batch of data from your data loader to test your code using `next(iter(bert_loader))`.

In [21]:
# train BERT neural network

epochs = 2
lr = 1e-4
optimizer = optim.Adam(bert_model.parameters(), lr=lr)

print("Starting BERT training...")

for epoch in range(epochs):
    
    epoch_loss = 0
    batch_iter = 0

    for batch in bert_loader:

        ids, mask, labels = batch

        # get predictions
        y_pred = bert_model(ids, mask)

        # calculate loss
        loss = bert_model.loss(y_pred, labels)

        # compute gradients in a backward pass
        loss.backward()

        # take a gradient step to update parameters
        optimizer.step()

        # zero gradients in preparation for next iteration
        optimizer.zero_grad()

        epoch_loss += loss.item()

        batch_iter += 1
        if batch_iter % 20 == 0:
            print('epoch: ', epoch, 'batch: ', batch_iter)

    print('epoch: ', epoch, 'loss:', f"{epoch_loss:.3}")
print("BERT training completed")

Starting BERT training...
epoch:  0 batch:  20
epoch:  0 batch:  40
epoch:  0 batch:  60
epoch:  0 batch:  80
epoch:  0 batch:  100
epoch:  0 batch:  120
epoch:  0 batch:  140
epoch:  0 batch:  160
epoch:  0 batch:  180
epoch:  0 batch:  200
epoch:  0 batch:  220
epoch:  0 batch:  240
epoch:  0 loss: 1.08e+02
epoch:  1 batch:  20
epoch:  1 batch:  40
epoch:  1 batch:  60
epoch:  1 batch:  80
epoch:  1 batch:  100
epoch:  1 batch:  120
epoch:  1 batch:  140
epoch:  1 batch:  160
epoch:  1 batch:  180
epoch:  1 batch:  200
epoch:  1 batch:  220
epoch:  1 batch:  240
epoch:  1 loss: 74.7
BERT training completed


In [22]:
## save model parameters
torch.save(bert_model.state_dict(), "recipe-distilbert.pt")

(b) Train the BOW neural network to predict vegetarian or not vegetarian.

In [23]:
## train neural network to predict vegetarian

epochs = 100
lr = 1e-4
optimizer = optim.Adam(bow_model.parameters(), lr=lr)

print("Starting BOW training...")

for epoch in range(epochs):
    
    epoch_loss = 0

    for x_batch, y_batch in bow_loader:

        y_pred = bow_model(x_batch)
        loss   = bow_model.loss(y_pred, y_batch)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        epoch_loss += loss.item()

    if epoch % 20 == 0:
        print('epoch: ', epoch, 'loss:', f"{epoch_loss:.3}")
print("BOW training completed")

Starting BOW training...
epoch:  0 loss: 86.3
epoch:  20 loss: 8.04
epoch:  40 loss: 0.994
epoch:  60 loss: 0.239
epoch:  80 loss: 0.137
BOW training completed


In [24]:
## save model parameters
torch.save(bow_model.state_dict(), "recipe-bow.pt")

### Question 7

(a) Calculate test accuracy for the BERT model. Note: this is done as a for loop as it requires too much memory to input all test data at once into the BERT model.

In [25]:
bert_model.eval()
bert_acc = 0
n_test = len(test_ind)

with torch.no_grad():
    for i in range(n_test):
        pred = bert_model(
            bert_ids_test[i].unsqueeze(0),
            bert_mask_test[i].unsqueeze(0)
        )
        pred = (pred >= 0.5).float()
        bert_acc += (pred == ybert_test[i]).sum() / len(pred)

bert_acc = bert_acc / n_test

(b) Calculate test accuracy for the BOW model. 

In [26]:
bow_model.eval()

with torch.no_grad():
    ybow_pred = bow_model(xbow_test)
    bow_acc = ((ybow_pred >= 0.5).float() == ybow_test).float().mean()

In [27]:
print("Final Results:")
print("BERT accuracy:", bert_acc.item())
print("BOW accuracy:", bow_acc.item())

Final Results:
BERT accuracy: 0.848976731300354
BOW accuracy: 0.7967537045478821


### Question 8

How do the BERT and BOW accuracies compare? Comment on the differences between the BERT and BOW models.

[Answer]:

In my results, BERT accuracy was 0.8489 and BOW accuracy was 0.7967. 
This means BERT performed better by about 5.2 percentage points.

The BOW model uses only the ingredient bag-of-words features. It checks whether each ingredient is present or not, so it is simple and fast. However, it does not understand word meaning, order, or context.

The BERT model uses the full recipe text, including the title, description, and ingredients. BERT can understand context better than BOW, which helps it make stronger predictions. For example, it can better understand differences like “chicken broth” versus “vegetable broth.”

Overall, BERT performed better because it uses richer text information and pre-trained language understanding. BOW still performed well because ingredients alone are very useful for predicting whether a recipe is vegetarian.

Just for fun: input your own recipe ingredients and see what the BERT model predicts.

In [28]:
my_recipe_collection = [
"The ingredients are: tofu bacon, bread, arugula, nutritional yeast",
"Recipe uses: beef, milk, eggs, ketchup, worchestershire sauce"
]

for recipe in my_recipe_collection:

    recipe_tokens = tokenizer(recipe, return_tensors='pt', 
                                max_length=180, padding='max_length', truncation=True)


    my_pred = bert_model(recipe_tokens['input_ids'], recipe_tokens['attention_mask'])
    my_pred = my_pred.detach().numpy()

    print(my_pred)


[0.9387374]
[0.00667408]
